In [51]:
# ============================================================
# STEP 1 — LOAD DATA & OVERVIEW
# Description:
# This script processes an EPC dataset to generate two new columns:
# 1) WATER_HEAT_CODE:
#    - Uses MAIN_HEAT_CODE if hot water is from main system
#    - Otherwise maps from a reference CSV
#    - Appends efficiency suffix (_1 to _5)
#
# 2) WATER_FUEL_CODE:
#    - Uses MAIN_FUEL_CODE if hot water is from main system
#    - Otherwise maps from a reference CSV
#
# Finally, the updated dataset is saved to a new CSV file.
# ============================================================

import pandas as pd

# File paths
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_fuel_codes_updated.csv"
ref_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/hot_water_code.csv"
output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_water_codes.csv"

# Load datasets
epc_df = pd.read_csv(epc_path)
ref_df = pd.read_csv(ref_path)

# Preview
print("EPC dataset shape:", epc_df.shape)
print(epc_df.head())

print("\nReference table:")
print(ref_df)

EPC dataset shape: (117, 203)
   BUILDING_REFERENCE_NUMBER  OSG_REFERENCE_NUMBER              ADDRESS1  \
0                 1001829067           122009933.0  19 CULCREUCH AVENUE    
1                 1001951858           122010025.0             GLENVIEW    
2                 1001829071           122009868.0  22 CULCREUCH AVENUE    
3                 1234761001           122009968.0    1 MENZIES TERRACE    
4                 1001991633           122009892.0       50 MAIN STREET    

  ADDRESS2  ADDRESS3 POSTCODE   LATITUDE  LONGITUDE INSPECTION_DATE  \
0  FINTRY   GLASGOW   G63 0YB  56.055692  -4.223337         9/29/25   
1  FINTRY   GLASGOW   G63 0XJ  56.052824  -4.220640         9/19/25   
2  FINTRY   GLASGOW   G63 0YB  56.055503  -4.223691         7/29/25   
3  FINTRY   GLASGOW   G63 0YJ  56.058427  -4.224838         7/22/25   
4  FINTRY   GLASGOW   G63 0XF  56.054738  -4.228307         7/17/25   

         TYPE_OF_ASSESSMENT  ... NUMBER_OF_OCCUPANTS  OCCUPANTS_ROUNDED_UP  \
0  RdSAP

In [52]:
# ============================================================
# STEP 2 (FIXED) — CLEAN + MAP EFFICIENCY VALUES
# Description:
# Fixes common issues preventing mapping:
# - Leading/trailing whitespace
# - Hidden characters (e.g. non-breaking spaces)
# - Inconsistent casing
# ============================================================

# Clean the column first
epc_df["HOT_WATER_ENERGY_EFF_CLEAN"] = (
    epc_df["HOT_WATER_ENERGY_EFF"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)  # remove non-breaking spaces
    .str.strip()                            # remove leading/trailing spaces
)

# Optional: standardise case (only if needed)
# epc_df["HOT_WATER_ENERGY_EFF_CLEAN"] = epc_df["HOT_WATER_ENERGY_EFF_CLEAN"].str.title()

# Mapping
eff_map = {
    "Very Poor": 1,
    "Poor": 2,
    "Average": 3,
    "Good": 4,
    "Very Good": 5
}

epc_df["HW_EFF_NUM"] = epc_df["HOT_WATER_ENERGY_EFF_CLEAN"].map(eff_map)

# Debugging output
print("Unique cleaned values:")
print(epc_df["HOT_WATER_ENERGY_EFF_CLEAN"].unique())

# Check for unmapped values
unmapped = epc_df[epc_df["HW_EFF_NUM"].isna()]

if not unmapped.empty:
    print("\n⚠️ Unmapped values found:")
    print(unmapped["HOT_WATER_ENERGY_EFF"].unique())
else:
    print("\n✅ All values mapped successfully")

Unique cleaned values:
['Poor' 'Average' 'Very Poor' 'Good']

✅ All values mapped successfully


In [53]:
# ============================================================
# STEP 3 (REPLACEMENT) — CLEAN HOTWATER_DESCRIPTION + LOOKUPS
# Description:
# Fixes mapping issues by cleaning HOTWATER_DESCRIPTION values:
# - Removes hidden characters
# - Strips whitespace
# - Standardises text
# ============================================================

# Clean HOTWATER_DESCRIPTION in EPC dataset
epc_df["HOTWATER_DESCRIPTION_CLEAN"] = (
    epc_df["HOTWATER_DESCRIPTION"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
)

# Clean HOTWATER_DESCRIPTION in reference table
ref_df["HOTWATER_DESCRIPTION_CLEAN"] = (
    ref_df["HOTWATER_DESCRIPTION"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
)

# Create lookup dictionaries using CLEAN column
heat_lookup = dict(zip(ref_df["HOTWATER_DESCRIPTION_CLEAN"], ref_df["WATER_HEAT_CODE"]))
fuel_lookup = dict(zip(ref_df["HOTWATER_DESCRIPTION_CLEAN"], ref_df["WATER_FUEL_CODE"]))

# Define cleaned "from main system" values
main_system_values = [
    "From main system",
    "From main system, no cylinder thermostat"
]

# Debug check
print("Unique cleaned HOTWATER_DESCRIPTION values:")
print(epc_df["HOTWATER_DESCRIPTION_CLEAN"].unique())

Unique cleaned HOTWATER_DESCRIPTION values:
['Electric immersion, off-peak' 'From main system'
 'From main system, no cylinder thermostat'
 'Electric immersion, standard tariff'
 'Electric instantaneous at point of use' 'Community scheme'
 'No system present: electric immersion assumed']


In [54]:
# ============================================================
# STEP 4 (REPLACEMENT) — CREATE BASE WATER HEAT CODE
# Description:
# Uses cleaned values and correctly handles "from main system"
# ============================================================

def get_base_water_heat_code(row):
    desc = row["HOTWATER_DESCRIPTION_CLEAN"]
    
    if desc in main_system_values:
        return row["MAIN_HEAT_CODE"]  # NO suffix later
    else:
        return heat_lookup.get(desc, None)

epc_df["WATER_HEAT_CODE_BASE"] = epc_df.apply(get_base_water_heat_code, axis=1)

# Debug missing mappings
missing_heat = epc_df[epc_df["WATER_HEAT_CODE_BASE"].isna()]
if not missing_heat.empty:
    print("Missing WATER_HEAT_CODE mappings:")
    print(missing_heat["HOTWATER_DESCRIPTION"].unique())

In [55]:
# ============================================================
# STEP 5 (FINAL FIX) — STRICT NO-SUFFIX FOR MAIN SYSTEM CASES
# Description:
# Ensures suffix is NEVER added for:
# - "From main system"
# - "From main system, no cylinder thermostat"
#
# Adds defensive cleaning + explicit boolean flag to avoid mismatch.
# ============================================================

# Create a clean + normalised comparison column (stronger cleaning)
epc_df["HOTWATER_DESCRIPTION_NORM"] = (
    epc_df["HOTWATER_DESCRIPTION"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
    .str.lower()
)

# Define normalised main system values
main_system_values_norm = [
    "from main system",
    "from main system, no cylinder thermostat"
]

# Create explicit boolean flag (KEY FIX)
epc_df["IS_MAIN_SYSTEM_HW"] = epc_df["HOTWATER_DESCRIPTION_NORM"].isin(main_system_values_norm)

# Build final WATER_HEAT_CODE
def build_water_heat_code(row):
    base_code = row["WATER_HEAT_CODE_BASE"]
    
    # If main system → return EXACT main heat code (no suffix)
    if row["IS_MAIN_SYSTEM_HW"]:
        return row["MAIN_HEAT_CODE"]
    
    # Otherwise → append suffix
    else:
        return f"{base_code}_{int(row['HW_EFF_NUM'])}"

epc_df["WATER_HEAT_CODE"] = epc_df.apply(build_water_heat_code, axis=1)

# Cleanup helper columns
epc_df.drop(columns=["WATER_HEAT_CODE_BASE", "HOTWATER_DESCRIPTION_NORM", "IS_MAIN_SYSTEM_HW"], inplace=True)

# Debug check
print("Check main system rows (should have NO suffix):")
print(
    epc_df.loc[
        epc_df["HOTWATER_DESCRIPTION"].str.contains("From main system", na=False),
        ["HOTWATER_DESCRIPTION", "MAIN_HEAT_CODE", "WATER_HEAT_CODE"]
    ].head()
)

Check main system rows (should have NO suffix):
                        HOTWATER_DESCRIPTION  \
2                          From main system    
3  From main system, no cylinder thermostat    
6  From main system, no cylinder thermostat    
7                          From main system    
8                          From main system    

                              MAIN_HEAT_CODE  \
2  air_source_heat_pump_radiators_electric_4   
3                     boiler_radiators_oil_3   
6  boiler_radiators_dual_fuel_mineral_wood_3   
7                     boiler_radiators_oil_3   
8  air_source_heat_pump_radiators_electric_4   

                             WATER_HEAT_CODE  
2  air_source_heat_pump_radiators_electric_4  
3                     boiler_radiators_oil_3  
6  boiler_radiators_dual_fuel_mineral_wood_3  
7                     boiler_radiators_oil_3  
8  air_source_heat_pump_radiators_electric_4  


In [56]:
# ============================================================
# STEP 6 (FULL REPLACEMENT) — ROBUST WATER FUEL CODE LOGIC
# Description:
# Fully rebuilds WATER_FUEL_CODE with:
# - Cleaned + normalised text matching (fixes mapping issues)
# - Explicit handling of "main system" cases
# - Safe lookup from reference table
# ============================================================

# --- CLEAN + NORMALISE EPC COLUMN ---
epc_df["HOTWATER_DESCRIPTION_NORM"] = (
    epc_df["HOTWATER_DESCRIPTION"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
    .str.lower()
)

# --- CLEAN + NORMALISE REFERENCE TABLE ---
ref_df["HOTWATER_DESCRIPTION_NORM"] = (
    ref_df["HOTWATER_DESCRIPTION"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
    .str.lower()
)

# --- CREATE LOOKUP DICTIONARY (NORMALISED KEYS) ---
fuel_lookup_norm = dict(
    zip(ref_df["HOTWATER_DESCRIPTION_NORM"], ref_df["WATER_FUEL_CODE"])
)

# --- DEFINE MAIN SYSTEM CONDITIONS (NORMALISED) ---
main_system_values_norm = [
    "from main system",
    "from main system, no cylinder thermostat"
]

# --- CREATE BOOLEAN FLAG ---
epc_df["IS_MAIN_SYSTEM_HW"] = epc_df["HOTWATER_DESCRIPTION_NORM"].isin(main_system_values_norm)

# --- BUILD WATER_FUEL_CODE ---
def build_water_fuel_code(row):
    desc = row["HOTWATER_DESCRIPTION_NORM"]
    
    # Case 1: From main system → use MAIN_FUEL_CODE directly
    if row["IS_MAIN_SYSTEM_HW"]:
        return row["MAIN_FUEL_CODE"]
    
    # Case 2: Lookup from reference table
    return fuel_lookup_norm.get(desc, None)

epc_df["WATER_FUEL_CODE"] = epc_df.apply(build_water_fuel_code, axis=1)

# --- DEBUGGING CHECKS ---
print("Unique NORMALISED descriptions (EPC):")
print(epc_df["HOTWATER_DESCRIPTION_NORM"].unique())

missing_fuel = epc_df[epc_df["WATER_FUEL_CODE"].isna()]
if not missing_fuel.empty:
    print("\n⚠️ Missing WATER_FUEL_CODE mappings:")
    print(missing_fuel["HOTWATER_DESCRIPTION"].unique())
else:
    print("\n✅ All WATER_FUEL_CODE values successfully mapped")

Unique NORMALISED descriptions (EPC):
['electric immersion, off-peak' 'from main system'
 'from main system, no cylinder thermostat'
 'electric immersion, standard tariff'
 'electric instantaneous at point of use' 'community scheme'
 'no system present: electric immersion assumed']

✅ All WATER_FUEL_CODE values successfully mapped


In [57]:
# ============================================================
# STEP 7 (UPDATED CLEANUP) — REMOVE TEMP COLUMNS
# Description:
# Cleans up helper columns created during processing
# ============================================================

cols_to_drop = [
    "HOTWATER_DESCRIPTION_NORM",
    "IS_MAIN_SYSTEM_HW"
]

# Drop only if they exist (safe cleanup)
epc_df.drop(columns=[c for c in cols_to_drop if c in epc_df.columns], inplace=True)

print("Cleanup complete")

Cleanup complete


In [58]:
# ============================================================
# STEP 7 — CLEANUP & SAVE OUTPUT
# Description:
# Remove helper columns and save final dataset.
# ============================================================

# Drop helper column
epc_df.drop(columns=["HW_EFF_NUM"], inplace=True)

# Save output
epc_df.to_csv(output_path, index=False)

print(f"File successfully saved to:\n{output_path}")

File successfully saved to:
/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_water_codes.csv
